# __1.Probelm__

- VGGNet에서 filter 크기를 줄이고, depth를 높여 표현력을 증가시키는 방법을 제공하였다. 그러나 이는 `gradient vanishing, training difficulty` 문제가 발생한다. 하지만 위 논문이 집중한것은 gradient vanishing이 아니다.
`gradient vanishing은 normalized initialization, intermediate normalization layers 방식을 통해 해결`
- (1)degradation problem
    - `identity mapping` 더 깊은 모델임에도 불구하고 training error가 높은것을 관찰하였다. 이는 overfitting과 gradient vanishing과는 관련이 없다. 왜냐하면, training error가 높았기 때문이다. 예를들어, 3번층 이후에 5개의 층을 더 쌓았을때, identity mapping이라면 결과는 이전층과의 결과와 유사하거나 좋아야하지만, 더 악화된 결과를 도출하였다.
    - `complex parameter space` 층이 깊어질수록 loss함수의 parameter 공간은 복잡해진다. 이런 복잡한 공간에서는 `local minimal` `saddle point`가 많다. 즉 optimizer 단계에서 진짜 최소값을 찾는것이 어려워지게 된다.

# __2.Contribution__

- 층이 깊어질수록 표현력이 좋아진다는것은 일부 사실이다. 그러나 이미 이전층에서 충분히 좋은 결과를 도찰하였다면, 뒤에 추가된 층들은 입력을 보내기만 하여도 괜찮은 상황이다(불필요한 연산 필요가 없다).

- ResNet Block : ResNet은 문제를 해결하기 위해 새로운 연산 `x + F(x)` 를 제시하였다. 
    - (1) x : 이전 layer의 feature 결과값이다. (edge detector)
    - (2) F(x) : F(x)는 resnet block이 학습하는것이다.(edge refinement)
    - `즉 H(x) = F(x) + x 는 이전 feature을 유지하면서 개선한 결과이다.`

- `H(x) = x ?` : H(x)가 x에 가까워졌다는 말은 feature의 개선의 필요도가 줄어들었다는 것이고, 학습이 충분히 완료된 상태에 가까워 졌음을 의미한다. 따라서 기존 CNN에서는 H(x), 이는 전체 feature을 다 계산하여 맞추어야한다. 그러나 resnet block을 통해 F(x) + x에서 F(x) 를 0에 가깝게 하도록 하면 학습이 훨씬 수월하게 진행된다. 이는 `네트워크가 전체 mapping을 학습하는 대신, 작은 수정값 F(x)만 학습하게 만들어 문제를 해결한다.`

# __3.ResNet Block__

- (1)input image : 224 x 224 x 3 의 이미지라고 가정하자.
- (2)first convolution : 첫 conv에서는 기존 CNN처럼 연산한다.
    - input -> conv -> pooling -> feature_map
- (3)ResNet Block : (2)를 거쳐 나온 x = feature_map부터 ResNet Bolck이 적용된다. `Block 내부는 conv(3x3) -> relu -> conv(3x3)이다.`
위 block을 통해 나온 결과를 `F(x)`라고 한다. 이때 Block의 `output = F(x) + x`이다. 
- (4) Repeat...
- (5) global average pooling
- (6) fully connected'